In [ ]:
from markov_models import MarkovModel 
from info_rate import compute_info_rate, count_ling_units
from helpers import update_values_in_csv, check_data_availability, load_config
from syllabification import parse_to_phones_and_sylls , get_largest_ipa_corpus
import numpy as np
import pickle
from pathlib import Path
import logging 

# NON PARRALLELIZED VERSION

logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)s] [%(process)d] %(message)s'
)
logger = logging.getLogger()

def run_pipeline(): 
    languages = ['DEU'] # ['JPN', 'VIE', 'YUE', 'ENG', 'FRA']

    for language in languages:
        config_dict = load_config(language)

        #parse_to_phones_and_sylls(language, config_dict)

        for processing_type in ['words']: 
            print(f"\nLanguage: {language}")
            print(f"Processing type: {processing_type.upper()}")
            print(f"=======================================================================================")

            #input_path = check_data_availability(language, processing_type, config_dict)
            
            if processing_type == 'words': 
                expected_corpus_size = config_dict['Corpus Size']
                exists, existing_path = get_largest_ipa_corpus(language, expected_corpus_size)
                if not exists:
                    logger.error(f"No IPA corpus found for {language} with size {expected_corpus_size}")
                    return None
                input_path = Path(existing_path)
            else: 
                folder = Path("produced_data") / language / processing_type
                filename = f"phonized_{language}.pkl" if processing_type == 'phones' else f"syllabified_{language}.pkl"
                input_path = folder / filename

            if input_path: 
                with open(input_path, "rb") as f:
                    data = pickle.load(f)
                    print(data[:5])
                
            else: continue
            
            for text_type in ['within_words', 'across_sentences']:
                if processing_type == 'words' and text_type == 'within_words': 
                    continue # Skip computing within word ngrams for text_type words

                print(f"\n📊 Computing ID and IR {' '.join(word.capitalize() for word in text_type.split('_'))}")


                n_values = [4]  # For bigram, trigram, and quadgram models
                markov_models = {}

                for n in n_values:

                    print(f"\n🧮 Training a Markov Model with n = {n}:")

                    # Create and build the Markov model
                    model = MarkovModel(n)

                    # Build the markov model
                    model.build(data, text_type)

                    # Compute the conditional entropy (information density)
                    info_density = model.compute_conditional_entropy()
                    print(f"Information Density: {info_density:.4f}")

                    # Compute the information rate (bits per second)
                    info_rate_values, speech_rate_values = compute_info_rate(info_density, processing_type, language)
                    print(f"Information Rate: {np.mean(info_rate_values):.4f}")
                    
                    # Update the CSV file with the computed values
                    update_values_in_csv(language, info_density, n, 'ID', text_type, processing_type)
                    update_values_in_csv(language, info_rate_values, n, 'IR', text_type, processing_type)
                    update_values_in_csv(language, speech_rate_values, n, 'SR', text_type, processing_type)
                    

                    # Store model for later use 
                    markov_models[n] = model

                    # Display exactly 3 examples
                    """example_count = 0
                    print("\nExample probabilities (p(x, y)):")

                    for (prefix, suffix), p_xy in model.cond_probs.items():
                        print(f"p({prefix} -> {suffix}) = {p_xy:.4f}")
                        example_count += 1
                        if example_count == 3:
                            break"""
                    
                    # Save the model to a file
                    model.save_model(language, processing_type, text_type)

            # For plotting, see plotting.ipynb

run_pipeline()

In [ ]:
from markov_models import MarkovModel 
from info_rate import compute_info_rate, count_ling_units
from helpers import update_values_in_csv, check_data_availability, load_config
from syllabification import parse_to_phones_and_sylls
import numpy as np
import pickle
from pathlib import Path
import logging
from itertools import product
from joblib import Parallel, delayed
import psutil

# PARALLELIZED VERSION

logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)s] [%(process)d] %(message)s'
)
logger = logging.getLogger()

def run_pipeline(language, processing_type, text_type, n_values):
    logging.info(f"🔍 Initial memory: {psutil.Process().memory_info().rss / 1e6:.2f} MB")
    config_dict = load_config(language)
   
    existing_ipa_path, phonized_path, syllabified_path, corpus_size_str, is_near_expected = parse_to_phones_and_sylls(language, config_dict)

    #input_path = check_data_availability(language, processing_type, config_dict)

    if processing_type == 'words': 
        input_path = existing_ipa_path
    else: 
        input_path = phonized_path if processing_type == 'phones' else syllabified_path

    if not input_path.exists():
        logger.error(f"Input path does not exist: {input_path}")
        return None
    with open(input_path, "rb") as f:
        data = pickle.load(f)
        
    if processing_type == 'words' and text_type == 'within_words':
        return 

    markov_models = {}
    # Create and build the Markov model
    for n in n_values:

        # Create and build the Markov model
        model = MarkovModel(n)

        # Build the markov model
        model.build(data, text_type)

        # Compute the conditional entropy (information density)
        info_density = model.compute_conditional_entropy()
        #logger.info(f"Information Density: {info_density:.4f}")

        # Compute the information rate (bits per second)
        info_rate_values, speech_rate_values = compute_info_rate(info_density, processing_type, language)
        logger.info(f"🧮 {language} | corpus size: {corpus_size_str} | {processing_type} | {text_type} | n={n} | IR: {np.mean(info_rate_values):.4f}")
        
        # Update the CSV file with the computed values for a corpus with largest posssible size 
        if is_near_expected: 
            update_values_in_csv(language, info_density, n, 'ID', text_type, processing_type)
            update_values_in_csv(language, info_rate_values, n, 'IR', text_type, processing_type)
            update_values_in_csv(language, speech_rate_values, n, 'SR', text_type, processing_type) 

            # Store model for later use 
            markov_models[n] = model
            
            # Save the model to a file
            model.save_model(language, processing_type, text_type, corpus_size_str)
    return True  


# Create all combinations to process, adjust as needed
languages = ['FRA', 'DEU', 'ENG']pr
processing_types = ['sylls']
text_types = ['across_sentences']
n_values = [4]  

tasks = list(product(languages, processing_types, text_types))

results = Parallel(n_jobs=4, verbose=5)( # Adjust n_jobs based on number of tasks
    delayed(run_pipeline)(lang, proc, txt, n_values)
    for lang, proc, txt in tasks
)

success_count = sum(r is True for r in results)
if success_count == 0:
    logging.error("No valid results. Please check the input data and configurations.")
else:
    logging.info(f"✅ {success_count} tasks completed successfully.")

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


✅ IPA corpus for DEU exists at produced_data/DEU/ipa_corpus_DEU_size:100.pkl. Skipping IPA generation.
IPA corpus has limited size. Expected/largest size:642700.
⏩ Skipping tokenization: phonemized and syllabified data already exist for DEU with size 100.


INFO:root:ngram examples: [('vas', 'pas', 'iːɾt', 'ɪn'), ('pas', 'iːɾt', 'ɪn', 'dɛɾ'), ('iːɾt', 'ɪn', 'dɛɾ', 'høː'), ('ɪn', 'dɛɾ', 'høː', 'lə'), ('ɪç', 'bɪn', 'nɔø', 'ɡiː'), ('bɪn', 'nɔø', 'ɡiː', 'rɪç'), ('nɔø', 'ɡiː', 'rɪç', 'ɪç'), ('ɡiː', 'rɪç', 'ɪç', 'hɑː'), ('rɪç', 'ɪç', 'hɑː', 'bə'), ('ɪç', 'hɑː', 'bə', 'kaɪ'), ('hɑː', 'bə', 'kaɪ', 'nə'), ('bə', 'kaɪ', 'nə', 'ɑː'), ('kaɪ', 'nə', 'ɑː', 'nʊŋ'), ('tsvaɪ', 'ʊn', 'dɾaɪs', 'ɪç'), ('ʊn', 'dɾaɪs', 'ɪç', 'mɑ')]


✅ IPA corpus for ENG exists at produced_data/ENG/ipa_corpus_ENG_size:100.pkl. Skipping IPA generation.
IPA corpus has limited size. Expected/largest size:440000.
⏩ Skipping tokenization: phonemized and syllabified data already exist for ENG with size 100.


INFO:root:ngram examples: [('maɪ', 'peə', 'ɹənts', 'wʊd'), ('peə', 'ɹənts', 'wʊd', 'ɹɪ'), ('ɹənts', 'wʊd', 'ɹɪ', 'pjuː'), ('wʊd', 'ɹɪ', 'pjuː', 'dɪ'), ('ɹɪ', 'pjuː', 'dɪ', 'eɪt'), ('pjuː', 'dɪ', 'eɪt', 'maɪ'), ('dɪ', 'eɪt', 'maɪ', 'bɹʌ'), ('eɪt', 'maɪ', 'bɹʌ', 'ðə'), ('maɪ', 'bɹʌ', 'ðə', 'ɪf'), ('bɹʌ', 'ðə', 'ɪf', 'ðeɪ'), ('ðə', 'ɪf', 'ðeɪ', 'ɛ'), ('ɪf', 'ðeɪ', 'ɛ', 'və'), ('ðeɪ', 'ɛ', 'və', 'faʊnd'), ('ɛ', 'və', 'faʊnd', 'aʊt'), ('və', 'faʊnd', 'aʊt', 'hiː')]
INFO:root:🧮 DEU | corpus size: 100 | sylls | across_sentences | n=4 | IR: 50.4416
INFO:root:🧮 ENG | corpus size: 100 | sylls | across_sentences | n=4 | IR: 50.8847
INFO:root:ngram examples: [('ʒə', 'ne', 'pa', 'fɛ'), ('ne', 'pa', 'fɛ', 'la'), ('pa', 'fɛ', 'la', 'di'), ('fɛ', 'la', 'di', 'fe'), ('la', 'di', 'fe', 'ʁɑ̃s'), ('di', 'fe', 'ʁɑ̃s', 'ɑ̃tʁ'), ('fe', 'ʁɑ̃s', 'ɑ̃tʁ', 'ø'), ('am', 'lɛt', 'a', 'ʒi'), ('lɛt', 'a', 'ʒi', 'kɔm'), ('a', 'ʒi', 'kɔm', 'sil'), ('ʒi', 'kɔm', 'sil', 'e'), ('kɔm', 'sil', 'e', 'tɛ'), ('sil', 'e', 'tɛ', 

✅ IPA corpus for FRA exists at produced_data/FRA/ipa_corpus_FRA_size:100.pkl. Skipping IPA generation.
IPA corpus has limited size. Expected/largest size:570550.
⏩ Skipping tokenization: phonemized and syllabified data already exist for FRA with size 100.


INFO:root:🧮 FRA | corpus size: 100 | sylls | across_sentences | n=4 | IR: 59.5541
[Parallel(n_jobs=4)]: Done   3 out of   3 | elapsed:    5.2s finished
INFO:root:✅ 3 tasks completed successfully.


In [ ]:
# Create all combinations to process, adjust as needed
languages = ['ENG']
processing_types = ['words']
text_types = ['across_sentences']
n_values = [3,4]  

tasks = list(product(languages, processing_types, text_types))

results = Parallel(n_jobs=4, verbose=5)( # Adjust n_jobs based on number of tasks
    delayed(run_pipeline)(lang, proc, txt, n_values)
    for lang, proc, txt in tasks
)

if all(r is None for r in results):
    logging.error("No valid results. Please check the input data and configurations.")

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
INFO:root:🧮 Training ENG | words | across_sentences | n=3
INFO:root:ngram examples: [('maɪ', 'peəɹənts', 'wʊd'), ('peəɹənts', 'wʊd', 'ɹɪpjuːdɪeɪt'), ('wʊd', 'ɹɪpjuːdɪeɪt', 'maɪ'), ('ɹɪpjuːdɪeɪt', 'maɪ', 'bɹʌðə'), ('maɪ', 'bɹʌðə', 'ɪf'), ('bɹʌðə', 'ɪf', 'ðeɪ'), ('ɪf', 'ðeɪ', 'ɛvə'), ('ðeɪ', 'ɛvə', 'faʊnd'), ('ɛvə', 'faʊnd', 'aʊt'), ('faʊnd', 'aʊt', 'hiː'), ('aʊt', 'hiː', 'wɒz'), ('hiː', 'wɒz', 'ɡeɪ'), ('ɪn', 'ɔːdə', 'tuː'), ('ɔːdə', 'tuː', 'kiːp'), ('tuː', 'kiːp', 'hɪz')]


In [ ]:
import cProfile
import pstats
from pstats import SortKey

def profile_run():
    profiler = cProfile.Profile()
    profiler.enable()
    
    # Run the pipeline for a single language (serially, no parallelism inside)
    run_pipeline()

    profiler.disable()

    # Dump to stats object and print top time-consuming lines
    stats = pstats.Stats(profiler).sort_stats(SortKey.CUMULATIVE)
    stats.print_stats(30)

profile_run()